# Segmentation using background subtraction

Use background substraction methods to properly segment the moving objects from their background. Use one of the videos with static camera.

Use the following approaches:
- Accumulated weighted image
- Mixture of Gaussian (MOG2)


In [4]:
import cv2 as cv
import numpy as np

In [5]:
# Video file names
file1 = 'data/crosswalk.avi'
file2 = 'data/fourway.avi'
file3 = 'data/night.avi'

### Accumulated weighted image

In [10]:
def accumulate_weighted_image(
        file: str,
        alpha: float = 0.02,
        scale: float = 0.5,
        threshold: int = 25,
        kernel_size: int = 3,
        morph_iter: int = 1,
        convert_to_gray: bool = False,
):
    # Create a kernel for morphological operations
    kernel = cv.getStructuringElement(cv.MORPH_RECT, (kernel_size, kernel_size))

    # Load the video
    cap = cv.VideoCapture(file)

    # Read the first frame
    _, frame = cap.read()

    # Convert the first frame to grayscale
    if convert_to_gray:
        frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

    # Initialize the background model
    bg_model = np.float32(frame)

    while cap.isOpened():
        # Read the next frame of the video
        success, frame = cap.read()
        # Exit the loop if the frame was not successfully read
        if not success:
            cv.waitKey(0)
            break

        # Convert the current frame to grayscale
        if convert_to_gray:
            frame = cv.cvtColor(frame, cv.COLOR_BGR2GRAY)

        # Update the background model using an accumulated weighted image
        cv.accumulateWeighted(frame, bg_model, alpha)

        # Subtract the background
        background = cv.convertScaleAbs(bg_model)
        diff = cv.absdiff(frame, background)
        # Apply a threshold to the difference image
        _, thresh = cv.threshold(diff, threshold, 255, cv.THRESH_BINARY)

        # Apply a morphological operations to the thresholded image
        if morph_iter > 0:
            thresh = cv.morphologyEx(thresh, cv.MORPH_CLOSE, kernel, iterations=morph_iter)
            thresh = cv.morphologyEx(thresh, cv.MORPH_OPEN, kernel, iterations=morph_iter)

        # Convert the thresholded image to 1 channel
        if not convert_to_gray:
            thresh = thresh[:, :, 0]

        # Extract the foreground
        foreground = cv.bitwise_and(frame, frame, mask=thresh)

        # Resize the images for display
        if scale != 1.0:
            frame = cv.resize(frame, (0, 0), fx=scale, fy=scale)
            background = cv.resize(background, (0, 0), fx=scale, fy=scale)
            thresh = cv.resize(thresh, (0, 0), fx=scale, fy=scale)
            foreground = cv.resize(foreground, (0, 0), fx=scale, fy=scale)

        # Display the resulting images
        cv.imshow('Frame', frame)
        cv.imshow('Thresh', thresh)
        cv.imshow('Background', background)
        cv.imshow('Foreground', foreground)

        # Exit the program if the user presses the 'q' key
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the VideoCapture and close all windows
    cap.release()
    cv.destroyAllWindows()

In [11]:
accumulate_weighted_image(
    file1,
    alpha=0.01,
    scale=0.5,
    threshold=15,
    kernel_size=5,
    morph_iter=1,
    convert_to_gray=False,
)

In [12]:
accumulate_weighted_image(
    file2,
    alpha=0.01,
    scale=0.5,
    threshold=20,
    kernel_size=5,
    morph_iter=1,
    convert_to_gray=False,
)

In [14]:
accumulate_weighted_image(
    file3,
    alpha=0.01,
    scale=0.5,
    threshold=10,
    kernel_size=5,
    morph_iter=1,
    convert_to_gray=False,
)

### Mixture of Gaussian (MOG2)

In [32]:
def all_background_subtractors(
        file: str,
        scale: float = 0.5,
        kernel_size: int = 3,
        morph_iter: int = 1,
):
    # Create kernel for morphological operation
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (kernel_size, kernel_size))

    # Capture frames from a camera
    cap = cv.VideoCapture(file)

    # Creating substractors objects for background subtraction
    fgbg_mog = cv.bgsegm.createBackgroundSubtractorMOG()
    fgbg_mog2 = cv.createBackgroundSubtractorMOG2()
    fgbg_gmg = cv.bgsegm.createBackgroundSubtractorGMG()
    fgbg_knn = cv.createBackgroundSubtractorKNN()

    # Read the frames from the camera one by one
    while cap.isOpened():
        # Read next frame
        success, frame = cap.read()
        # Exit if frame not successfully read
        if not success:
            cv.waitKey(0)
            break

        # Resize the frame
        img = cv.resize(frame, (0, 0), fx=scale, fy=scale)

        # Apply mask for background subtraction
        fg_mask_mog = fgbg_mog.apply(img)
        fg_mask_mog2 = fgbg_mog2.apply(img)
        fg_mask_gmg = fgbg_gmg.apply(img)
        fg_mask_knn = fgbg_knn.apply(img)

        # Morphological operation
        if morph_iter > 0:
            # Close
            fg_mask_mog_morph = cv.morphologyEx(fg_mask_mog, cv.MORPH_CLOSE, kernel, iterations=morph_iter)
            fg_mask_mog2_morph = cv.morphologyEx(fg_mask_mog2, cv.MORPH_CLOSE, kernel, iterations=morph_iter)
            fg_mask_gmg_morph = cv.morphologyEx(fg_mask_gmg, cv.MORPH_CLOSE, kernel, iterations=morph_iter)
            fg_mask_knn_morph = cv.morphologyEx(fg_mask_knn, cv.MORPH_CLOSE, kernel, iterations=morph_iter)
            # Open
            fg_mask_mog_morph = cv.morphologyEx(fg_mask_mog_morph, cv.MORPH_OPEN, kernel, iterations=morph_iter)
            fg_mask_mog2_morph = cv.morphologyEx(fg_mask_mog2_morph, cv.MORPH_OPEN, kernel, iterations=morph_iter)
            fg_mask_gmg_morph = cv.morphologyEx(fg_mask_gmg_morph, cv.MORPH_OPEN, kernel, iterations=morph_iter)
            fg_mask_knn_morph = cv.morphologyEx(fg_mask_knn_morph, cv.MORPH_OPEN, kernel, iterations=morph_iter)

            cv.imshow('MOG Morph', fg_mask_mog_morph)
            cv.imshow('MOG2 Morph', fg_mask_mog2_morph)
            cv.imshow('GMG Morph', fg_mask_gmg_morph)
            cv.imshow('KNN Morph', fg_mask_knn_morph)

        # Display the resulting frame
        cv.imshow('Original', img)
        cv.imshow('MOG', fg_mask_mog)
        cv.imshow('MOG2', fg_mask_mog2)
        cv.imshow('GMG', fg_mask_gmg)
        cv.imshow('KNN', fg_mask_knn)

        # Exit if the user presses the 'q' key
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the VideoCapture and close all windows
    cap.release()
    cv.destroyAllWindows()

In [35]:
all_background_subtractors(file1, scale=0.4, kernel_size=3, morph_iter=1)

In [41]:
def background_subtractor(
        file: str,
        subtractor: str,
        scale: float = 0.5,
        kernel_size: int = 3,
        morph_iter: int = 1,
):
    # Creating object for background subtraction
    subtractor = subtractor.upper()
    if subtractor == 'MOG':
        fgbg = cv.bgsegm.createBackgroundSubtractorMOG()
    elif subtractor == 'MOG2':
        fgbg = cv.createBackgroundSubtractorMOG2()
    elif subtractor == 'GMG':
        fgbg = cv.bgsegm.createBackgroundSubtractorGMG()
    elif subtractor == 'KNN':
        fgbg = cv.createBackgroundSubtractorKNN()
    else:
        raise ValueError('Unknown background subtractor')

    # Create a kernel for morphological operation
    kernel = cv.getStructuringElement(cv.MORPH_RECT, (kernel_size, kernel_size))

    # Capture frames from a camera
    cap = cv.VideoCapture(file)

    # Read the frames from the camera
    while cap.isOpened():
        # Read next frame
        success, frame = cap.read()
        # Exit if frame not successfully read
        if not success:
            cv.waitKey(0)
            break

        # Resize the frame
        img = cv.resize(frame, (0, 0), fx=scale, fy=scale)

        # Apply mask for background subtraction
        fg_mask = fgbg.apply(img)

        # Apply morphological operation
        if morph_iter > 0:
            fg_mask = cv.morphologyEx(fg_mask, cv.MORPH_CLOSE, kernel)
            fg_mask = cv.morphologyEx(fg_mask, cv.MORPH_OPEN, kernel)

        # Extract the foreground and background
        foreground = cv.bitwise_and(img, img, mask=fg_mask)
        background = fgbg.getBackgroundImage()

        # Display the resulting frame
        cv.imshow('Frame', img)
        cv.imshow(f'FG Mask ({subtractor})', fg_mask)
        cv.imshow(f'Foreground', foreground)
        cv.imshow(f'Background', background)

        # Exit if the user presses the 'q' key
        if cv.waitKey(1) & 0xFF == ord('q'):
            break

    # Release the VideoCapture and close all windows
    cap.release()
    cv.destroyAllWindows()

In [42]:
background_subtractor(file1, 'mog2', scale=0.4, kernel_size=3, morph_iter=2)

In [160]:
background_subtractor(file2, 'mog2', scale=0.4, kernel_size=3, morph_iter=2)

In [ ]:
background_subtractor(file3, 'mog2', scale=0.4, kernel_size=3, morph_iter=2)